## Header

In [1]:
print("=" * 55)
print("  Smart Health AI — Model Training (Notebook)")
print("=" * 55)

  Smart Health AI — Model Training (Notebook)


## Import Libraries

In [2]:
import os, time, warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import joblib
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score

## Load Dataset

In [3]:
# ── 1. Load Dataset ──────────────────────────────────────
print("\n[1/7] Loading dataset ...")
t0 = time.time()

# dataset comes as TWO files — Training.csv and Testing.csv
# Both have 132 binary symptom columns + 1 label column called "prognosis"
TRAIN_PATH = "Training.csv"
TEST_PATH  = "Testing.csv"

for path in [TRAIN_PATH, TEST_PATH]:
    if not os.path.exists(path):
        raise FileNotFoundError(
            f"\n❌ File not found: {path}"
            f"\n   Download from: kaggle.com/datasets/kaushil268/disease-prediction-using-machine-learning"
            f"\n   Place Training.csv and Testing.csv in the same folder as this notebook."
        )

df_train = pd.read_csv(TRAIN_PATH)
df_test  = pd.read_csv(TEST_PATH)

print(f"   Loaded in {time.time()-t0:.1f}s")
print(f"   Training set : {df_train.shape[0]} rows, {df_train.shape[1]} columns")
print(f"   Testing set  : {df_test.shape[0]} rows,  {df_test.shape[1]} columns")


[1/7] Loading dataset ...
   Loaded in 0.1s
   Training set : 4920 rows, 134 columns
   Testing set  : 42 rows,  133 columns


## Check Dataset Shape

In [4]:
df_train.shape
df_train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4920 entries, 0 to 4919
Columns: 134 entries, itching to Unnamed: 133
dtypes: float64(1), int64(132), object(1)
memory usage: 5.0+ MB


## Clean Dataset

In [5]:
# ── 2. Clean Dataset ─────────────────────────────────────
# Strip whitespace from column names and prognosis values
df_train.columns = df_train.columns.str.strip()
df_test.columns  = df_test.columns.str.strip()

df_train["prognosis"] = df_train["prognosis"].str.strip()
df_test["prognosis"]  = df_test["prognosis"].str.strip()

# Remove the unnamed last column that appears in Testing.csv
df_train = df_train.loc[:, ~df_train.columns.str.contains("^Unnamed")]
df_test  = df_test.loc[:, ~df_test.columns.str.contains("^Unnamed")]

print("After cleaning:")
print(f"   Training set : {df_train.shape}")
print(f"   Testing set  : {df_test.shape}")
print(f"   Unique diseases: {df_train['prognosis'].nunique()}")
print("\nDisease list:")
for i, d in enumerate(sorted(df_train["prognosis"].unique()), 1):
    print(f"   {i:2}. {d}")

After cleaning:
   Training set : (4920, 133)
   Testing set  : (42, 133)
   Unique diseases: 41

Disease list:
    1. (vertigo) Paroymsal  Positional Vertigo
    2. AIDS
    3. Acne
    4. Alcoholic hepatitis
    5. Allergy
    6. Arthritis
    7. Bronchial Asthma
    8. Cervical spondylosis
    9. Chicken pox
   10. Chronic cholestasis
   11. Common Cold
   12. Dengue
   13. Diabetes
   14. Dimorphic hemmorhoids(piles)
   15. Drug Reaction
   16. Fungal infection
   17. GERD
   18. Gastroenteritis
   19. Heart attack
   20. Hepatitis B
   21. Hepatitis C
   22. Hepatitis D
   23. Hepatitis E
   24. Hypertension
   25. Hyperthyroidism
   26. Hypoglycemia
   27. Hypothyroidism
   28. Impetigo
   29. Jaundice
   30. Malaria
   31. Migraine
   32. Osteoarthristis
   33. Paralysis (brain hemorrhage)
   34. Peptic ulcer diseae
   35. Pneumonia
   36. Psoriasis
   37. Tuberculosis
   38. Typhoid
   39. Urinary tract infection
   40. Varicose veins
   41. hepatitis A


## Prepare Features

In [6]:
# ── 3. Prepare Features & Labels ─────────────────────────
X_train = df_train.drop("prognosis", axis=1).astype("int8")
X_test  = df_test.drop("prognosis",  axis=1).astype("int8")

# Align columns — make sure test has same columns as train in same order
X_test = X_test.reindex(columns=X_train.columns, fill_value=0)

# ADD ENCODER
label_encoder = LabelEncoder()
y_train = label_encoder.fit_transform(df_train["prognosis"])
y_test  = label_encoder.transform(df_test["prognosis"])

# Save the symptom column names (used by app.py)
symptom_columns = X_train.columns.tolist()

print(f"Feature matrix (train) : {X_train.shape}")
print(f"Feature matrix (test)  : {X_test.shape}")
print(f"Target classes         : {len(label_encoder.classes_)}")

Feature matrix (train) : (4920, 132)
Feature matrix (test)  : (42, 132)
Target classes         : 41


## Train Model

In [7]:
# ── 5. Train RandomForestClassifier ──────────────────────
#
#  n_estimators=200   — 200 trees is more than enough for 42 clean classes
#  max_depth=None     — grow trees fully; dataset is small (4920 rows) and clean
#  min_samples_leaf=1 — allow single-sample leaves; balanced dataset so no noise risk
#  class_weight=None  — dataset is perfectly balanced (~120 rows per disease)
#  n_jobs=-1          — use ALL CPU cores in parallel

print(f"Training rows   : {X_train.shape[0]}")
print(f"Test rows       : {X_test.shape[0]} (Testing.csv)")

print("\n[5/7] Training RandomForestClassifier ...")
t1 = time.time()

model = RandomForestClassifier(
    n_estimators=200,
    max_depth=None,
    min_samples_leaf=1,
    max_features="sqrt",
    class_weight=None,
    random_state=42,
    n_jobs=-1,
    verbose=1
)

model.fit(X_train, y_train)
train_time = time.time() - t1
print(f"\n   ✅ Training done in {train_time:.1f}s")

Training rows   : 4920
Test rows       : 42 (Testing.csv)

[5/7] Training RandomForestClassifier ...


[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 12 concurrent workers.
[Parallel(n_jobs=-1)]: Done  26 tasks      | elapsed:    0.0s



   ✅ Training done in 0.7s


[Parallel(n_jobs=-1)]: Done 176 tasks      | elapsed:    0.3s
[Parallel(n_jobs=-1)]: Done 200 out of 200 | elapsed:    0.4s finished


## Check Accuracy

In [8]:
# ── 6. Evaluate ──────────────────────────────────────────
train_acc = model.score(X_train, y_train)
test_acc  = model.score(X_test,  y_test)

print(f"Training Accuracy   : {train_acc*100:.2f}%")
print(f"Test Accuracy       : {test_acc*100:.2f}%")

y_pred = model.predict(X_test)
report = classification_report(y_test, y_pred, target_names=label_encoder.classes_, output_dict=True)

macro_f1 = report["macro avg"]["f1-score"]
w_f1     = report["weighted avg"]["f1-score"]
print(f"Macro F1-score         : {macro_f1*100:.2f}%")
print(f"Weighted F1-score      : {w_f1*100:.2f}%")

[Parallel(n_jobs=12)]: Using backend ThreadingBackend with 12 concurrent workers.
[Parallel(n_jobs=12)]: Done  26 tasks      | elapsed:    0.0s
[Parallel(n_jobs=12)]: Done 176 tasks      | elapsed:    0.2s
[Parallel(n_jobs=12)]: Done 200 out of 200 | elapsed:    0.2s finished
[Parallel(n_jobs=12)]: Using backend ThreadingBackend with 12 concurrent workers.
[Parallel(n_jobs=12)]: Done  26 tasks      | elapsed:    0.0s
[Parallel(n_jobs=12)]: Done 176 tasks      | elapsed:    0.0s
[Parallel(n_jobs=12)]: Done 200 out of 200 | elapsed:    0.0s finished
[Parallel(n_jobs=12)]: Using backend ThreadingBackend with 12 concurrent workers.
[Parallel(n_jobs=12)]: Done  26 tasks      | elapsed:    0.0s
[Parallel(n_jobs=12)]: Done 176 tasks      | elapsed:    0.0s
[Parallel(n_jobs=12)]: Done 200 out of 200 | elapsed:    0.0s finished


Training Accuracy   : 100.00%
Test Accuracy       : 97.62%
Macro F1-score         : 98.37%
Weighted F1-score      : 97.62%


## Save Model

In [9]:
# ── 7. Save Artifacts ────────────────────────────────────
print("\n── Saving model artifacts ──")
MODEL_DIR = "model"
os.makedirs(MODEL_DIR, exist_ok=True)

# compress=3 reduces .pkl file size by 5-10x with minimal speed penalty
joblib.dump(model,           os.path.join(MODEL_DIR, "disease_model.pkl"),  compress=3)
joblib.dump(label_encoder,   os.path.join(MODEL_DIR, "label_encoder.pkl"),  compress=3)
joblib.dump(symptom_columns, os.path.join(MODEL_DIR, "model_features.pkl"), compress=3)


print(f"   ✅ disease_model.pkl")
print(f"   ✅ label_encoder.pkl")
print(f"   ✅ model_features.pkl")


── Saving model artifacts ──


   ✅ disease_model.pkl
   ✅ label_encoder.pkl
   ✅ model_features.pkl


## Verify

In [10]:
loaded_model    = joblib.load(os.path.join(MODEL_DIR, "disease_model.pkl"))
loaded_encoder  = joblib.load(os.path.join(MODEL_DIR, "label_encoder.pkl"))
loaded_features = joblib.load(os.path.join(MODEL_DIR, "model_features.pkl"))

print(f"Model classes   : {len(loaded_model.classes_)}")
print(f"Encoder classes : {len(loaded_encoder.classes_)}")
assert len(loaded_model.classes_) == len(loaded_encoder.classes_)
print("✅ Verification successful")

Model classes   : 41
Encoder classes : 41
✅ Verification successful


## Test Prediction

In [11]:
# ── 8. Sanity Check Prediction ───────────────────────────
sample = dict.fromkeys(loaded_features, 0)

# Example symptoms — should predict Fungal infection
sample["itching"]              = 1
sample["skin_rash"]            = 1
sample["nodal_skin_eruptions"] = 1

test_df = pd.DataFrame([sample]).astype("int8")

probs   = loaded_model.predict_proba(test_df)[0]
top3_idx = np.argsort(probs)[-3:][::-1]

print("Top Predictions (itching + skin_rash + nodal_skin_eruptions):")
for i in top3_idx:
    disease_name = loaded_encoder.inverse_transform([i])[0]
    print(f"   {disease_name}: {probs[i]*100:.2f}%")

print(f"\n✅ All done!  Total time: {time.time()-t0:.1f}s")

[Parallel(n_jobs=12)]: Using backend ThreadingBackend with 12 concurrent workers.
[Parallel(n_jobs=12)]: Done  26 tasks      | elapsed:    0.0s
[Parallel(n_jobs=12)]: Done 176 tasks      | elapsed:    0.1s
[Parallel(n_jobs=12)]: Done 200 out of 200 | elapsed:    0.1s finished


Top Predictions (itching + skin_rash + nodal_skin_eruptions):
   Fungal infection: 100.00%
   Varicose veins: 0.00%
   Urinary tract infection: 0.00%

✅ All done!  Total time: 2.2s


## System Info

In [12]:
import sys
print(sys.executable)

d:\SMART_HEALTH_AI\.venv\Scripts\python.exe


In [13]:
import os
print(os.path.exists("model/disease_model.pkl"))

True
